## LLM model
`model = ModelInterface(`
    `model_id,`
    `prams,`
    `credentials,`
    `project_id`
`)`

`model_id = 'meta-llama/llma-302-90b-vision-instruct'`
-- the model is an instruct model, since the name has '-insturct'

`parameters = {GenParams.MAS_NEW_TOKEN: 256,GenParams.TEMPREATURE: 0.2}`

`credentials = {"url": "https://s-south.ml.could.ibm.com"}`

`project_id = "skills-network"`

- to run the model 
    - `model.generate()`
    - `print(msg['result][0]['generated_text'])`


## Chat Model
- force the model to LangChain compatable
    - text-in, text-out model as expected by the LangChain.
    - `WhatsonxLLM(model)`
    - `print(llama_ll.invoke("who is man's best friend?"))`

## Groq

In [ ]:
import os

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

GROQ_API_KEY = os.getenv("GROQ")

llm = ChatGroq(
    model="llama-3.3-70b-versatile"
    , temprature=0
    , api_key=GROQ_API_KEY
)

embedding_model = HuggingFaceEmbeddings(
    model_name = "all-MiniLM-L6-v2"
)

: 

## Google

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage

import os


gemini_api = os.getenv("GEMINI")

# 1. Initialize the Chat Model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
    max_tokens=500,
    google_api_key=gemini_api
)

embedding_model = GoogleGenerativeAIEmbeddings(
    model='models/text-embedding-004'
    , task_type="retrieval_document"
    , google_api_key=gemini_api
)



## Chat messages
- `SystemMessage` 
    - have high weightage, it has presidence over the Human message
    - During RHLF models are thught that the system messages are Master rules
    - Set the presona and the bounderies
- `HumanMessage`
    - task description
- `AIMessage`
    - LLM's reponse to the HumanMessage
    - Can be used for "One-shot" or "Few-shot" learning

`from langchain_cre.messages import HuamMessage, SystemMessage, AIMessage`



## Prompt Templates
- input parameters to the model

### String prompt templates
- to format a single string
- for simple inputs

In [3]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "Tell me one {adjective} joke about {topic}"
    )

input_ = {
    "adjective": "funny",
    "topic" : "cats"
}

prompt.invoke(input_)

StringPromptValue(text='Tell me one funny joke about cats')

### Chat prompt templates
- designed to work with chat models
- can assign various roles to the messates
    - system
    - human
    - ai

In [1]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "Tell me a joke about{topic}")
])

input_ = {"topic" : "cates"}

prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me a joke aboutcates', additional_kwargs={}, response_metadata={})])


### MessagePlaceholder
- Special tool for ChatPromptTemplate
- dynamic container for a list of messages
- standard place holder like {topic} expect a single string
- MessagePlaceholder excepts an array of message objects like HumanMessage or AIMessage

#### purpose
- can inject entire history of messages into a template
- code below

|Feature|{variable_name} (String)|MessagesPlaceholder|
|---|---|---|
|Expected Data | A single string. | A list of Message objects.|
|Result | Replaces text inside a message. | Adds multiple messages to the list.|
|Best For,"Keywords | topics, names." | "Chat history, Agent scratches, memory."|

In [5]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    # This is where the magic happens:
    MessagesPlaceholder(variable_name="chat_history"), 
    ("human", "{input}"),
])

# When you invoke this, you pass a LIST of messages for "chat_history"
# and a STRING for "input".

ModuleNotFoundError: No module named 'langchain.prompts'

## Output parsers
- conver the output of the LLM to a more suitable form
    - like to CSV or Json

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

import os


gemini_api = os.getenv("GEMINI")

# 1. Initialize the Chat Model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
    max_tokens=500,
    google_api_key=gemini_api
)

In [11]:
# JSON Parser
from langchain_core.output_parsers import JsonOutputParser

# Need ot import a BaseModel and Field form langchain to generate the model output
from pydantic import BaseModel, Field

class Joke(BaseModel):
    setup: str = Field(description = "Question to setup a joke")
    punchline: str = Field(description = "Answer to the joke")


joke_query = "Tell me a joke"

output_parser = JsonOutputParser(pydantic_object=Joke)

format_instructions = output_parser.get_format_instructions()

prompt = PromptTemplate(
    input_variables = ["joke_query"],
    template = "Generate a joke in JSON format with the following fields: {format_instructions}\n\n{joke_query}",
    partial_variables = {"format_instructions": format_instructions}
)

chain = prompt | llm | output_parser

chain.invoke({"joke_query": joke_query})


{'setup': "Why don't scientists trust atoms?",
 'punchline': 'Because they make up everything!'}

- need format instructions
- says LLMs to produce output in a pariticular way that is understandable to the output parser
- in PromptTemplate
    - partial_variables = {"format_insturctions": output_parser.get_fromat_instructions()}

- class passed to the output parser
    - define the schema

    ``` python
    class Movie(BaseModel):
            "name":str = Field(description="the name of the movie")
            # define a field 'name' in output with datatype as str
            "released_year":int = Field(description="the year at which the movie has released")
    ```

- this class has to be passed to the outputpaser so that it came to know about the output of the llm output
- now we need to force the llm to generate output accourding to this schema
    - done through the prompt

```python
    prompt = ChatPrompt(
        template = "Answer the user query.\n{format_instructions}\n{query}\n"
        , input_varibales=['query']
        , partial_variables={"format_instructions":outputparser.get_format_instrutions()}
    )

```


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field

import os


gemini_api = os.getenv("GEMINI")

# 1. Initialize the Chat Model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
    max_tokens=500,
    google_api_key=gemini_api
)

class Movie(BaseModel):
    title:str = Field(description="movie title")
    director:str = Field(description="movie director")
    year:int = Field(description="movie released year")
    genre:str = Field(description="movie genre")

json_parser = JsonOutputParser(pydantic_object=Movie)

prompt_template = PromptTemplate(
    template = "Generate information about the movie.\n{format_instructions}\n{movie_name}\n"
    , input_varibales=['movie_name']
    , partial_variables={"format_instructions":json_parser.get_format_instructions()}
)

movie_chain = prompt_template | llm | json_parser

# test with a movie name
movie_name = "The Matrix"
result = movie_chain.invoke({"movie_name": movie_name})

print(result)



{'title': 'The Matrix', 'director': 'The Wachowskis', 'year': 1999, 'genre': 'Science Fiction'}


## Documents
- Document object contain infroamtion about some data
- 2 attributes
    - page_content:str
    - menta_data:dict

```python
from langchain_core.documents import Document

Document(page_content="""
Python is an interpretted high-level genenral purpose programming language.
"""
metadata={
    "document_id": 2020
    , "docuemnt_source": "About_python"
    , "created_time": 1680013019
}
)
```

## Document loaders
- to load document from a variety of sources
    - eg pdf
- Langchain offers 100 distinct sources
- integration with other major providers
    - AirByte, Unstructured, Amazon S3 buckets

### PDF loader
- load pdf documents

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader

# loader = PyPDFLoader(
#     "./Documents/LangChain Basics.pdf"
#     , extract_images=False
#     )
loader = PyMuPDFLoader(
    "./Documents/LangChain Basics.pdf"
    , extract_images=False
    )

document = loader.load()

# to look at a page
document[2]

Document(metadata={'producer': 'Skia/PDF m143', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36', 'creationdate': '2026-01-16T09:06:50+00:00', 'source': './Documents/LangChain Basics.pdf', 'file_path': './Documents/LangChain Basics.pdf', 'total_pages': 56, 'format': 'PDF 1.4', 'title': 'Skills Network Labs', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-01-16T09:06:50+00:00', 'trapped': '', 'modDate': "D:20260116090650+00'00'", 'creationDate': "D:20260116090650+00'00'", 'page': 2}, page_content='Setup\nFor this lab, you will use the following libraries:\nibm-watson-ai , ibm-watson-machine-learning  for using LLMs from\nIBM\'s watsonx.ai.\nlangchain , langchain-ibm , langchain-community , langchain-\nexperimental  for using relevant features from LangChain.\npypdf  is an open-source pure-python PDF library capable of splitting, merging,\ncropping, and transforming the pages of PDF files.\nchromadb  is an o

In [3]:
document[1].page_content[:1000]

"In this lab, you will gain hands-on experience using LangChain to simplify the complex\nprocesses required to integrate advanced AI capabilities into practical applications. You\nwill apply core LangChain framework capabilities and use Langchain's innovative\nfeatures to build more intelligent, responsive, and efficient applications.\nTable of contents\n1. Objectives\n2. Setup\nA. Installing required libraries\nB. Importing required libraries\n3. LangChain concepts\nA. Model\nB. Chat model\nC. Chat message\na. Exercise 1: Compare Model Responses with Different Parameters\nD. Prompt templates\nE. Output parsers\na. Exercise 2: Creating and Using a JSON Output Parser\nF. Documents\na. Exercise 3: Working with Document Loaders and Text Splitters\nb. Exercise 4: Building a Simple Retrieval System with LangChain\nG. Memory\na. Exercise 5: Building a Chatbot with Memory using LangChain\nH. Chains\na. Exercise 6: Implementing Multi-Step Processing with Different Chain\nApproaches\nI. Tools a

## Text Splitter
- split the documents to manageable chunks. to fit to models context window
- working
    - split the text into meaningful chunks (often sentences)
    - combine the above chunks to certain size (as measured by a specific function)
    - then start creating a next chunk, with some overlap, to keep the context between the chunks

### Some text splitters
- CharacterTextSplitter
    - split by characters. measures chunk lentgh by number of characters

In [2]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")

chunks = text_splitter.split_documents(document)

print(len(chunks))

713


In [4]:
# to take a look at the page content
chunks[5].page_content

'interactions with LLMs. Data scientists can dynamically compare prompts and switch\nbetween foundation models without significant code modifications. These capabilities'

## Embedding models
- generate vector representation of chunks
- semantic search can be performed on this vectors
- one can specify the purpose embeddings

### Task types
|Task Type|Usage|
|---|---|
|RETRIEVAL_QUERY|Optimized for short user question|
|RETRIEVAL_DOCUMENT|Optimized for large blocks of searchable text|
|SEMANTIC_SIMILARITY|Best for comparing if two strings mean the same thing|


In [ ]:
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings

gemini_api = os.getenv("GEMINI")

embedding_model = GoogleGenerativeAIEmbeddings(
    model='models/text-embedding-004'
    , task_type="retrieval_document"
    , google_api_key=gemini_api
)

texts = [text.page_content for text in chunks]

embedding_result = embedding_model.embed_documents(texts)
embedding_result[0]

[0.00665653,
 -0.026778936,
 -0.045609027,
 0.0014161806,
 0.023130806,
 0.008418884,
 0.03373465,
 -0.016791815,
 0.020080348,
 0.042548522,
 -0.005672341,
 0.0140526,
 0.039516717,
 0.0026658091,
 -0.010700434,
 -0.047749564,
 0.023608563,
 0.03677314,
 -0.08153242,
 0.03877114,
 0.024119072,
 -0.028291628,
 -0.050841395,
 -0.05424177,
 0.00849343,
 -0.0005243845,
 0.036947247,
 0.029546307,
 0.00058978715,
 -0.040417172,
 0.012301926,
 0.050000094,
 0.009811688,
 -0.062875785,
 -0.00019976383,
 0.0006244626,
 0.04847164,
 0.019516034,
 0.04745189,
 -0.067448065,
 -0.02482002,
 0.021977141,
 -0.010150155,
 0.0708075,
 0.0040558483,
 -0.005313352,
 -0.03605113,
 0.004861062,
 -0.037316535,
 0.0215464,
 0.075100794,
 0.014173465,
 -0.04672312,
 0.012999744,
 -0.07540319,
 0.0015304333,
 -0.04325932,
 -0.00730391,
 0.013826546,
 0.049046088,
 -0.030122366,
 -0.040012438,
 -0.036402162,
 0.01811324,
 -0.008904702,
 -0.020517109,
 -0.011273851,
 -0.043738786,
 -0.016959246,
 0.020142589,


## Vector store
- to store the embeddings
- Many vector stores
    - Chroma

`from langchain.vectorstores import Chroma`

- to search the data from the Chroma db

```python
docsearch = Chroma.from_documents(chunks, embedding_model)
# perform the embeddings ont he chunks and stroe the data in a vector store
````

- to perform a similarity search and retrieve the docuemnt

```python
query = "what is Langchain?"
doc = docsearch.similarity_search(query)
print(doc[0].page_content)
```


In [7]:
import os

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma

document_path = "./Documents/LangChain Basics.pdf"

#lodading
loader = PyMuPDFLoader(
    document_path
    , extract_images=False
)
document = loader.load()

# splitting
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")
chunks = text_splitter.split_documents(document)

document_store = Chroma.from_documents(
    chunks
    , embedding_model
)

query = "what is langchain?"

docs = document_store.similarity_search(
    query
    , k = 3
)
print(docs[0].page_content)

IBM's watsonx.ai.
langchain , langchain-ibm , langchain-community , langchain-
experimental  for using relevant features from LangChain.


In [8]:
for doc in docs:
    print(doc.page_content)
    print('-------------------------')

IBM's watsonx.ai.
langchain , langchain-ibm , langchain-community , langchain-
experimental  for using relevant features from LangChain.
-------------------------
LangChain provides a lot of utilities for adding memory to a system. These 
utilities can be used by themselves or 
incorporated seamlessly into a chain.
-------------------------
LangChain provides a lot of utilities for adding memory to a system. These 
utilities can be used by themselves or 
incorporated seamlessly into a chain.
-------------------------


## Retrivers
- interface to return docuemnts using unstructured query
- may not be able to store documents
- can use a vector store as the backbone of a retriever
    - other type of retrivers are also exist

In [3]:
retriever = document_store.as_retriever(
    search_kwargs={"k":3} # pass the number of results
)
docs = retriever.invoke(query)

for doc in docs:
    print(doc.page_content)
    print('-------------------------')

large language models (LLMs). LangChain stands out by providing essential tools and
abstractions that enhance the customization, accuracy, and relevance of the
information generated by these models.
-------------------------
needed to complete various tasks. When integrated with LangChain, the LLM becomes a
powerful tool, providing the foundational structure necessary for building and
-------------------------
n home pageLangChain + LangGraphSearch...⌘KSupportGitHubTry LangSmithTry L
angSmithSearch...NavigationLangChain overviewLangChainLangGraphDeep Agents
-------------------------


## Parent document retrievers
- you have conflicting goals when you retrieve documents
    - small documents has embeddings that reflects the exact meaning of the content (precision)
    - the document has to be long enough to retain the context of the text (context)
- `ParentDocumentRetriever` strikes the balance
- during retreival, first featches the small chunks, however, look up the Parent IDs to retrieve the large document
- Stores data in 2 diff places
    - vector store (child chunks)
        - 100 - 200 words
        - used for actual search
    - Document Store (Parent Document)
        - entire original document
        - Stored in simple databases (like dictionary or InMemoryStore)
        - Linked to the children
        - Passes the Parent Document to the LLM

In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

parent_splitter = CharacterTextSplitter(
    chunk_size=2000
    , chunk_overlap=200
    , separator="\n"
)
child_splitter = CharacterTextSplitter(
    chunk_size=200
    , chunk_overlap=20
    , separator="\n"
)
vectorstore = Chroma(
    collection_name="split_parents"
    , embedding_function=embedding_model
)
# large memory store
store = InMemoryStore

retriver = ParentDocumentRetriver(
    vectorstore=vectorstore
    , document_store=store
    , child_splitter=child_splitter
    , parent_splitter=parent_splitter
)

# add documents to the hierarchical retrieval system
retriever.add_documents(document)

: 

In [ ]:
import os
os._exit(0)

: 